# CS781 Phase 3: Prefix tuning on pre-trained Selective SSM (Mamba-2 style) legal LM 

**How this notebook runs**
- Use **Save Version -> Save & Run All (Commit)** with *GPU T4 x2*, so training keeps going even if the browser closes.
- Sections 4-5 (download + tokenizer + tokenization) run once. Tokenizer, token files and checkpoints are written to `/kaggle/working/model3_ssm_output`.
- Training (section 8) runs as a 2-GPU DDP job. After `SESSION_BUDGET_HOURS` it saves a checkpoint and stops cleanly; the evaluation cells then skip themselves.
- **To continue in a new session:** new version -> *Add Input -> Notebook Output Files -> this notebook* -> Run All. The tokenized corpus, tokenizer and newest checkpoint are found automatically (nothing is re-downloaded or re-tokenized) and the data order resumes exactly where it stopped.
- When the single pass over the corpus is finished, sections 9-10 evaluate the best checkpoint and build the submission zip.
- All artifacts of this notebook carry an `ssm` tag in their file names, so attaching the output of the decoder-only / encoder-decoder notebooks as well can never be mistaken for SSM files.

# 1. Login

In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


# 2. Check GPU

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


# 3. Configuration

In [35]:
# =============================================================================
# 3. Configuration (Experiment 2: SSM + Prefix Tuning)
# =============================================================================
import os
import time
import torch
import glob
SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = torch.cuda.is_available()

# -----------------------------------------------------------------------------
# Phase 1 SSM Base Model Hyperparameters (Must match Phase 1 exactly)
# -----------------------------------------------------------------------------
VOCAB_SIZE = 32000
D_MODEL = 768
N_LAYERS = 12
D_STATE = 64          # mamba2 uses 64
EXPAND = 2
HEADDIM = 64
SSM_VARIANT = "mamba2"
SCAN_CHUNK = 64
GRAD_CKPT = False
# -----------------------------------------------------------------------------
# Phase 3 Task & Prefix Tuning Hyperparameters
# -----------------------------------------------------------------------------
NUM_LABELS = 13
PREFIX_LEN = 16              # Length of the trainable prefix vector

# Fine-Tuning Hyperparameters
NUM_EPOCHS = 10
MICRO_BATCH_SIZE = 16        # Increased because the base model is frozen (saves VRAM)
GLOBAL_BATCH_SAMPLES = 64
LEARNING_RATE = 1e-3         # Prefix tuning requires a slightly higher LR than full fine-tuning
MIN_LR = 1e-5
WEIGHT_DECAY = 0.1

# -----------------------------------------------------------------------------
# File Paths (Reusing datasets generated in Experiment 1)
# -----------------------------------------------------------------------------
# Dynamically find the mounted dataset folder to bypass Kaggle's hidden paths
def find_data_dir():
    hits = glob.glob("/kaggle/input/**/rr_train.pt", recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    raise FileNotFoundError("Could not find rr_train.pt. Please check if the dataset is attached.")

DATA_DIR = find_data_dir()
OUTPUT_DIR = "/kaggle/working/exp2_ssm_prefix"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")

TRAIN_DATA_PATH = os.path.join(DATA_DIR, "rr_train.pt")
VAL_DATA_PATH = os.path.join(DATA_DIR, "rr_val.pt")
TEST_DATA_PATH = os.path.join(DATA_DIR, "rr_test.pt")
LABEL_VOCAB_FILE = os.path.join(DATA_DIR, "label_vocab.json")

CONFIG_PATH = os.path.join(OUTPUT_DIR, "train_config.json")
STATUS_PATH = os.path.join(OUTPUT_DIR, "train_status.json")
LOG_PATH = os.path.join(OUTPUT_DIR, "train_log.jsonl")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Session Management
SESSION_BUDGET_HOURS = 8.0
DATALOADER_WORKERS = 2
try:
    import psutil
    SESSION_START = psutil.Process(os.getpid()).create_time()
except Exception:
    SESSION_START = time.time()

print(f"Experiment 2 Config Loaded | Device: {device} | Output Dir: {OUTPUT_DIR}")

Experiment 2 Config Loaded | Device: cuda | Output Dir: /kaggle/working/exp2_ssm_prefix


# 7. Model

In [36]:
%%writefile /kaggle/working/exp2_ssm_prefix/model_ssm.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization."""
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        var = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(var + self.eps) * self.weight


def selective_scan(dt, A, B, C, x, chunk: int = 64):
    """
    h_t = exp(dt_t * A) * h_{t-1} + dt_t * B_t * x_t ;  y_t = sum_n C_t[n] * h_t[:, n]

    Same maths as a step-by-step loop over all L positions, but done in two levels so that
    only `chunk + L/chunk` python-level steps are needed instead of L (2,048 -> ~96):
      1. inside every chunk, run the recurrence from h=0, all chunks in parallel;
      2. a short sequential pass over the chunks produces the state entering each chunk;
      3. add that carried state's contribution, exp(cumsum(dt*A)) * carry, to every position.
    Every exponent is <= 0 (A < 0, dt >= 0), so nothing can overflow.
    All inputs must be float32.  dt, x: (b, L, d)   A: (d, n)   B, C: (b, L, n)  ->  y: (b, L, d)
    """
    b, L, d = x.shape
    n = A.shape[1]
    pad = (-L) % chunk
    if pad:  # dt = 0 -> decay 1 and no input: the state is left untouched by the padded steps
        dt = F.pad(dt, (0, 0, 0, pad))
        x = F.pad(x, (0, 0, 0, pad))
        B = F.pad(B, (0, 0, 0, pad))
        C = F.pad(C, (0, 0, 0, pad))
    nc = (L + pad) // chunk
    dt = dt.reshape(b, nc, chunk, d)
    x = x.reshape(b, nc, chunk, d)
    B = B.reshape(b, nc, chunk, n)
    C = C.reshape(b, nc, chunk, n)

    dtA = dt.unsqueeze(-1) * A                                   # (b, nc, T, d, n)
    dA = torch.exp(dtA)
    dBx = dt.unsqueeze(-1) * x.unsqueeze(-1) * B.unsqueeze(3)    # (b, nc, T, d, n)

    # 1. local scan inside every chunk (starting from zero state)
    h = x.new_zeros(b, nc, d, n)
    ys = []
    for t in range(chunk):
        h = dA[:, :, t] * h + dBx[:, :, t]
        ys.append((h * C[:, :, t].unsqueeze(2)).sum(-1))         # (b, nc, d)
    y_local = torch.stack(ys, dim=2)                             # (b, nc, T, d)

    # 2. state entering each chunk
    S = torch.cumsum(dtA, dim=2)                                 # log of the decay accumulated inside the chunk
    P_end = torch.exp(S[:, :, -1])                               # (b, nc, d, n)
    carry = x.new_zeros(b, d, n)
    carries = []
    for c in range(nc):
        carries.append(carry)
        carry = P_end[:, c] * carry + h[:, c]
    carry_in = torch.stack(carries, dim=1)                       # (b, nc, d, n)

    # 3. contribution of the carried state to every position of its chunk
    y_corr = (torch.exp(S) * carry_in.unsqueeze(2) * C.unsqueeze(3)).sum(-1)   # (b, nc, T, d)

    y = (y_local + y_corr).reshape(b, nc * chunk, d)
    return y[:, :L]


class SelectiveSSM(nn.Module):
    """
    Selective State-Space Module (Mamba core): input-dependent (Delta, B, C) and a linear recurrence over positions.
    """
    def __init__(self, d_model: int = 768, d_state: int = 16, expand: int = 2,
                 dt_rank: int = None, kernel_size: int = 4, scan_chunk: int = 64):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_inner = d_model * expand
        self.dt_rank = dt_rank if dt_rank is not None else math.ceil(d_model / 16)
        self.scan_chunk = scan_chunk

        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=kernel_size,
                                padding=kernel_size - 1, groups=self.d_inner, bias=True)
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + self.d_state * 2, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        A = torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        self.reset_dt_parameters()

    def reset_dt_parameters(self):
        """Mamba's timescale init: dt in [1e-3, 1e-1] (log-uniform), stored through the inverse softplus."""
        dt_init_std = self.dt_rank ** -0.5
        nn.init.uniform_(self.dt_proj.weight, -dt_init_std, dt_init_std)
        dt_min, dt_max = 1e-3, 1e-1
        dt = torch.exp(torch.rand(self.d_inner) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min))
        dt = dt.clamp(min=1e-4)
        inv_softplus_dt = dt + torch.log(-torch.expm1(-dt))
        with torch.no_grad():
            self.dt_proj.bias.copy_(inv_softplus_dt)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq_len, _ = x.shape

        xz = self.in_proj(x)
        x_branch, z_branch = xz.chunk(2, dim=-1)

        x_conv = x_branch.transpose(1, 2)
        x_conv = self.conv1d(x_conv)[:, :, :seq_len]       # drop the right-hand causal padding
        x_act = F.silu(x_conv.transpose(1, 2))

        x_ssm = self.x_proj(x_act)
        dt_raw, B, C = torch.split(x_ssm, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt_raw))

        # The recurrence always runs in float32 (exp/cumsum/products are not safe in fp16 autocast).
        with torch.autocast(device_type=x.device.type, enabled=False):
            A = -torch.exp(self.A_log.float())
            x32 = x_act.float()
            y = selective_scan(dt.float(), A, B.float(), C.float(), x32, chunk=self.scan_chunk)
            y = y + x32 * self.D
            y = y * F.silu(z_branch.float())
        return self.out_proj(y.to(z_branch.dtype))


def segsum(x: torch.Tensor) -> torch.Tensor:
    """(..., T) -> (..., T, T) with out[i, j] = sum(x[j+1..i]) for i >= j and -inf above the diagonal."""
    T = x.size(-1)
    c = torch.cumsum(x, dim=-1)
    out = c.unsqueeze(-1) - c.unsqueeze(-2)
    return out.masked_fill(~torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device)), float("-inf"))


def ssd_scan(x, dt, A, B, C, chunk: int = 64):
    """
    Chunked state-space-duality scan (Mamba-2): same recurrence as a sequential loop,
        h_t = exp(A * dt_t) * h_{t-1} + dt_t * (x_t outer B_t) ;   y_t = h_t . C_t
    but computed with a handful of batched matrix multiplications instead of L (or L * d_inner * d_state) tiny steps:
      1. inside every chunk: (C B^T) * decay-mask  ->  matmul with x          (the diagonal blocks)
      2. the state each chunk leaves behind                                   (matmul)
      3. a short recurrence over the chunk states                             (tiny)
      4. the carried state's contribution to every position of its chunk      (matmul)
    All exponents are <= 0 (A < 0, dt >= 0), so nothing can overflow. Inputs must be float32.
    x: (b, L, h, p)   dt: (b, L, h)   A: (h,)   B, C: (b, L, n) shared by all heads   ->   y: (b, L, h, p)
    """
    b, L, h, p = x.shape
    n = B.size(-1)
    pad = (-L) % chunk
    if pad:  # dt = 0 -> decay 1 and no input: padded steps leave the state untouched
        x = F.pad(x, (0, 0, 0, 0, 0, pad))
        dt = F.pad(dt, (0, 0, 0, pad))
        B = F.pad(B, (0, 0, 0, pad))
        C = F.pad(C, (0, 0, 0, pad))
    nc = (L + pad) // chunk

    X = (x * dt.unsqueeze(-1)).reshape(b, nc, chunk, h, p)
    Bc = B.reshape(b, nc, chunk, n)
    Cc = C.reshape(b, nc, chunk, n)
    Ad = (A.view(1, 1, h) * dt).reshape(b, nc, chunk, h).permute(0, 3, 1, 2)          # (b, h, nc, chunk)
    Acs = torch.cumsum(Ad, dim=-1)

    # 1. diagonal blocks
    Lm = torch.exp(segsum(Ad))                                                         # (b, h, nc, l, s)
    G = torch.einsum("bcln,bcsn->bcls", Cc, Bc)
    Y_diag = torch.einsum("bhcls,bcshp->bclhp", G.unsqueeze(1) * Lm, X)

    # 2. state left behind by each chunk
    decay_states = torch.exp(Acs[..., -1:] - Acs)                                      # (b, h, nc, l)
    states = torch.einsum("bcln,bclhp->bchpn", Bc, X * decay_states.permute(0, 2, 3, 1).unsqueeze(-1))

    # 3. recurrence over chunk states (state entering chunk c = decayed sum of all earlier chunk states)
    states = torch.cat([torch.zeros_like(states[:, :1]), states], dim=1)               # (b, nc+1, h, p, n)
    decay_chunk = torch.exp(segsum(F.pad(Acs[..., -1], (1, 0))))                       # (b, h, nc+1, nc+1)
    states = torch.einsum("bhzc,bchpn->bzhpn", decay_chunk, states)[:, :-1]            # (b, nc, h, p, n)

    # 4. carried state -> outputs
    Y_off = torch.einsum("bcln,bchpn->bclhp", Cc, states) * torch.exp(Acs).permute(0, 2, 3, 1).unsqueeze(-1)

    return (Y_diag + Y_off).reshape(b, nc * chunk, h, p)[:, :L]


class SSDMixer(nn.Module):
    """
    Mamba-2 style selective state-space mixer: input-dependent (dt, B, C), a scalar decay A per head, heads of size
    `headdim`, B and C shared by all heads, gated RMS-norm before the output projection.
    Its scan is matmul-based (see ssd_scan), which is what makes it practical to train without custom CUDA kernels.
    """
    def __init__(self, d_model: int = 768, d_state: int = 64, expand: int = 2, headdim: int = 64,
                 kernel_size: int = 4, scan_chunk: int = 64):
        super().__init__()
        self.d_inner = d_model * expand
        assert self.d_inner % headdim == 0, "d_model * expand must be divisible by headdim"
        self.d_state = d_state
        self.headdim = headdim
        self.nheads = self.d_inner // headdim
        self.scan_chunk = scan_chunk
        conv_dim = self.d_inner + 2 * d_state                       # x, B, C go through the causal conv together

        self.in_proj = nn.Linear(d_model, 2 * self.d_inner + 2 * d_state + self.nheads, bias=False)   # z, x, B, C, dt
        self.conv1d = nn.Conv1d(conv_dim, conv_dim, kernel_size=kernel_size, padding=kernel_size - 1,
                                groups=conv_dim, bias=True)
        self.dt_bias = nn.Parameter(torch.zeros(self.nheads))
        self.A_log = nn.Parameter(torch.zeros(self.nheads))
        self.D = nn.Parameter(torch.ones(self.nheads))
        self.norm = RMSNorm(self.d_inner)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.reset_dt_parameters()

    def reset_dt_parameters(self):
        """dt in [1e-3, 1e-1] (log-uniform) through the inverse softplus; A in [1, 16] (log-uniform in magnitude)."""
        dt_min, dt_max = 1e-3, 1e-1
        dt = torch.exp(torch.rand(self.nheads) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)).clamp(min=1e-4)
        with torch.no_grad():
            self.dt_bias.copy_(dt + torch.log(-torch.expm1(-dt)))
            self.A_log.copy_(torch.log(torch.empty(self.nheads).uniform_(1.0, 16.0)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, L, _ = x.shape
        z, xBC, dt = torch.split(self.in_proj(x), [self.d_inner, self.d_inner + 2 * self.d_state, self.nheads], dim=-1)
        xBC = F.silu(self.conv1d(xBC.transpose(1, 2))[:, :, :L].transpose(1, 2))     # drop the right-hand causal padding
        xs, Bm, Cm = torch.split(xBC, [self.d_inner, self.d_state, self.d_state], dim=-1)

        # The scan always runs in float32 (exp / cumsum / products are not safe in fp16 autocast).
        with torch.autocast(device_type=x.device.type, enabled=False):
            dt = F.softplus(dt.float() + self.dt_bias)
            A = -torch.exp(self.A_log.float())
            xh = xs.float().reshape(b, L, self.nheads, self.headdim)
            y = ssd_scan(xh, dt, A, Bm.float(), Cm.float(), chunk=self.scan_chunk)
            y = (y + xh * self.D.view(1, 1, -1, 1)).reshape(b, L, self.d_inner)
            y = self.norm(y * F.silu(z.float()))
        return self.out_proj(y.to(z.dtype))


class MambaBlock(nn.Module):
    """Residual block with RMS normalisation around either mixer ("mamba2" = SSDMixer, "mamba1" = SelectiveSSM)."""
    def __init__(self, d_model: int, d_state: int = 64, expand: int = 2, scan_chunk: int = 64,
                 variant: str = "mamba2", headdim: int = 64):
        super().__init__()
        self.norm = RMSNorm(d_model)
        if variant == "mamba2":
            self.ssm = SSDMixer(d_model=d_model, d_state=d_state, expand=expand, headdim=headdim, scan_chunk=scan_chunk)
        elif variant == "mamba1":
            self.ssm = SelectiveSSM(d_model=d_model, d_state=d_state, expand=expand, scan_chunk=scan_chunk)
        else:
            raise ValueError(f"unknown SSM variant: {variant}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.ssm(self.norm(x))


class Model3_SSM(nn.Module):
    """Selective state-space causal language model."""
    def __init__(self, vocab_size: int = 32000, d_model: int = 768, n_layers: int = 12,
                 d_state: int = 64, expand: int = 2, scan_chunk: int = 64, grad_ckpt: bool = False,
                 variant: str = "mamba2", headdim: int = 64):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.grad_ckpt = grad_ckpt

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            MambaBlock(d_model=d_model, d_state=d_state, expand=expand, scan_chunk=scan_chunk,
                       variant=variant, headdim=headdim)
            for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embedding.weight      # weight tying

        self.apply(self._init_weights)
        # self.apply() above also hits the mamba1 dt_proj Linear (weights -> N(0, 0.02), bias -> 0) and would wipe
        # the timescale init: put it back (the mamba2 mixer keeps dt_bias / A_log as plain parameters, harmless there).
        for m in self.modules():
            if isinstance(m, (SelectiveSSM, SSDMixer)):
                m.reset_dt_parameters()

    @staticmethod
    def _init_weights(module: nn.Module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    def forward(self, input_ids: torch.Tensor, targets: torch.Tensor = None, last_only: bool = False):
        x = self.embedding(input_ids)
        for layer in self.layers:
            if self.grad_ckpt and self.training and torch.is_grad_enabled():
                # recompute the block in backward: the (b, L, d_inner, d_state) scan tensors are never kept for all layers
                x = checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        x = self.norm_f(x)
        if last_only:                    # generation: only the last position's logits are needed
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.vocab_size), targets.view(-1), ignore_index=0)  # 0 = <pad>
        return logits, loss

Overwriting /kaggle/working/exp2_ssm_prefix/model_ssm.py


In [37]:
%%writefile /kaggle/working/exp2_ssm_prefix/model_ssm_prefix.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class PrefixTunedSSMForRR(nn.Module):
    def __init__(self, base_model, n_layers=12, d_model=768, prefix_len=16, num_labels=13):
        super().__init__()
        self.base_model = base_model
        self.prefix_len = prefix_len
        self.num_labels = num_labels
        
        # 1. Freeze the entire pre-trained Phase 1 SSM
        for param in self.base_model.parameters():
            param.requires_grad = False
            
        # 2. Trainable Prefix Parameters: [N_Layers, Prefix_Length, D_Model]
        # Initialized from a normal distribution to break symmetry
        self.prefixes = nn.Parameter(torch.randn(n_layers, prefix_len, d_model))
        
        # 3. Task-specific classification head for Rhetorical Roles
        self.classifier = nn.Linear(d_model, num_labels)
        
    def forward(self, input_ids, end_indices, labels=None, class_weights=None):
        batch_size = input_ids.size(0)
        
        # Extract initial embeddings from the frozen SSM embedding layer
        x = self.base_model.embedding(input_ids)
        
        # Process through each Mamba layer with injected prefixes
        for i, layer in enumerate(self.base_model.layers):
            # Expand the learned prefix for this specific layer across the current batch
            # Shape: [Batch, Prefix_Length, D_Model]
            layer_prefix = self.prefixes[i].unsqueeze(0).expand(batch_size, -1, -1)
            
            # Prepend the prefix to the hidden states before the layer processes them
            # Shape becomes: [Batch, Prefix_Length + Sequence_Length, D_Model]
            x_concat = torch.cat([layer_prefix, x], dim=1)
            
            # Pass the concatenated sequence through the SSM block
            x_out = layer(x_concat)
            
            # Truncate the prefix positions so the sequence length matches input_ids
            # This prevents the prefix from accumulating across layers and misaligning end_indices
            x = x_out[:, self.prefix_len:, :]
            
        x = self.base_model.norm_f(x)
        
        # Extract sentence representations using end_indices (Shape: [Batch, Num_Sentences])
        num_sentences = end_indices.size(1)
        batch_indices = torch.arange(batch_size, device=x.device).unsqueeze(1).expand(-1, num_sentences)
        
        # sentence_representations shape: [Batch, Num_Sentences, D_Model]
        sentence_representations = x[batch_indices, end_indices, :]
        
        # Generate classification logits
        logits = self.classifier(sentence_representations)
        
        loss = None
        if labels is not None:
            # Flatten the batch for loss computation, ignoring padded sentences (-1)
            active_loss = labels.view(-1) != -1
            active_logits = logits.view(-1, self.num_labels)[active_loss]
            active_labels = labels.view(-1)[active_loss]
            
            if active_logits.numel() > 0:
                loss = F.cross_entropy(active_logits, active_labels, weight=class_weights)
            else:
                loss = torch.tensor(0.0, device=x.device, requires_grad=True)
            
        return logits, loss

Overwriting /kaggle/working/exp2_ssm_prefix/model_ssm_prefix.py


In [38]:
import sys
import gc
import math
import importlib
import torch
import torch.nn.functional as F

sys.path.insert(0, OUTPUT_DIR)
import model_ssm
import model_ssm_prefix
importlib.reload(model_ssm)
importlib.reload(model_ssm_prefix)

from model_ssm import Model3_SSM, selective_scan, ssd_scan
from model_ssm_prefix import PrefixTunedSSMForRR

# 1. Define kwargs for the Phase 1 Base Model
MODEL_KWARGS = dict(vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_layers=N_LAYERS, d_state=D_STATE, expand=EXPAND,
                    scan_chunk=SCAN_CHUNK, grad_ckpt=GRAD_CKPT, variant=SSM_VARIANT, headdim=HEADDIM)

print("=" * 70)
print("EXPERIMENT 2: PREFIX-TUNED SSM ARCHITECTURE SUMMARY")
print("=" * 70)

# 2. Instantiate Base Model and Prefix Wrapper
base_probe = Model3_SSM(**MODEL_KWARGS)
probe = PrefixTunedSSMForRR(base_model=base_probe, n_layers=N_LAYERS, d_model=D_MODEL, 
                            prefix_len=PREFIX_LEN, num_labels=NUM_LABELS)

# Verify parameter freezing
n_params_base = sum(p.numel() for p in base_probe.parameters())
n_params_trainable = sum(p.numel() for p in probe.parameters() if p.requires_grad)

print(f"Base Model (Frozen) Parameters: {n_params_base / 1e6:.1f}M")
print(f"Prefix + Head (Trainable) Parameters: {n_params_trainable / 1e6:.3f}M")
print(f"Trainable %: {(n_params_trainable / (n_params_base + n_params_trainable)) * 100:.2f}%\n")

# 3. Forward Pass Check with simulated Phase 3 Data
_b, _seq_len = 2, 70
_ids = torch.randint(4, VOCAB_SIZE, (_b, _seq_len))

# Mock end_indices (e.g., 3 sentences per document, pointing to specific token indices)
_end_indices = torch.tensor([[20, 45, 69], [15, 50, 69]])

# Mock labels (13 rhetorical role classes) and dummy class weights
_labels = torch.randint(0, NUM_LABELS, (_b, 3))
_class_weights = torch.ones(NUM_LABELS)

with torch.no_grad():
    _logits, _loss = probe(_ids, _end_indices, labels=_labels, class_weights=_class_weights)

print(f"Forward check | input {_ids.shape}, end_idx {_end_indices.shape} -> logits {_logits.shape} | loss {_loss.item():.3f}")
print("=" * 70)

del base_probe, probe, _logits, _loss
gc.collect()

EXPERIMENT 2: PREFIX-TUNED SSM ARCHITECTURE SUMMARY
Base Model (Frozen) Parameters: 68.6M
Prefix + Head (Trainable) Parameters: 0.157M
Trainable %: 0.23%

Forward check | input torch.Size([2, 70]), end_idx torch.Size([2, 3]) -> logits torch.Size([2, 3, 13]) | loss 2.933


70

# 8. Dataloaders

In [39]:
%%writefile /kaggle/working/exp2_ssm_prefix/data_rr.py
import torch
from torch.utils.data import Dataset

class RRSentenceDataset(Dataset):
    """
    Loads the pre-tokenized and aligned .pt files generated during Experiment 1.
    Each item contains 'input_ids', 'end_indices', and 'labels'.
    """
    def __init__(self, pt_path):
        self.data = torch.load(pt_path, weights_only=False, map_location="cpu")
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "id": item.get("id", str(idx)),
            "input_ids": item["input_ids"],
            "end_indices": item["end_indices"],
            "labels": item.get("labels", None)
        }

def compute_class_weights(dataset, num_labels=13):
    """
    Computes inverse frequency class weights to handle highly imbalanced
    legal rhetorical roles (e.g., FACT vs. STATUTE).
    """
    all_labels = []
    for item in dataset:
        if item.get("labels") is not None:
            lbls = item["labels"].view(-1)
            all_labels.append(lbls[lbls != -1])
            
    if not all_labels:
        return torch.ones(num_labels)
        
    all_labels = torch.cat(all_labels)
    counts = torch.bincount(all_labels, minlength=num_labels).float()
    counts = counts.clamp(min=1.0)
    
    weights = 1.0 / counts
    weights = weights / weights.sum() * num_labels
    return weights

Overwriting /kaggle/working/exp2_ssm_prefix/data_rr.py


In [40]:
import os
print(os.listdir("/kaggle/input"))

['datasets']


In [41]:
import os
import sys
import torch
sys.path.insert(0, OUTPUT_DIR)
from data_rr import RRSentenceDataset, compute_class_weights

# Load datasets from the Phase3feed254 mounted directory
train_dataset = RRSentenceDataset(TRAIN_DATA_PATH)
val_dataset = RRSentenceDataset(VAL_DATA_PATH)
test_dataset = RRSentenceDataset(TEST_DATA_PATH)

# Compute inverse frequency class weights for loss balancing
class_weights = compute_class_weights(train_dataset, num_labels=NUM_LABELS)

# Save class weights to OUTPUT_DIR so the DDP script can load them easily
torch.save(class_weights, os.path.join(OUTPUT_DIR, "class_weights.pt"))

# Calculate dynamic step counts based on Phase 3 data and Epochs
MAX_TRAIN_STEPS = (len(train_dataset) // GLOBAL_BATCH_SAMPLES) * NUM_EPOCHS
WARMUP_STEPS = int(MAX_TRAIN_STEPS * 0.1)  # 10% warmup is standard for Prefix Tuning

print("=" * 70)
print("EXPERIMENT 2: DATASET & PREFIX TUNING SCHEDULE SUMMARY")
print("=" * 70)
print(f"Train samples: {len(train_dataset):,} | Val samples: {len(val_dataset):,} | Test samples: {len(test_dataset):,}")
print(f"Global Batch Size: {GLOBAL_BATCH_SAMPLES} | Micro Batch: {MICRO_BATCH_SIZE}")
print(f"Total Epochs: {NUM_EPOCHS} | Max Train Steps: {MAX_TRAIN_STEPS:,} | Warmup Steps: {WARMUP_STEPS}")
print(f"Class Weights (First 5): {class_weights[:5].tolist()}")
print("=" * 70)

EXPERIMENT 2: DATASET & PREFIX TUNING SCHEDULE SUMMARY
Train samples: 64 | Val samples: 16 | Test samples: 10
Global Batch Size: 64 | Micro Batch: 16
Total Epochs: 10 | Max Train Steps: 10 | Warmup Steps: 1
Class Weights (First 5): [0.003816131502389908, 0.13942191004753113, 0.015411676838994026, 0.05709659680724144, 0.03700705245137215]


# 9. Training Loop

Training script (written to disk and launched with `torchrun` on both T4s; resumable, time-boxed).

In [42]:
%%writefile /kaggle/working/model3_ssm_output/train_ddp.py
"""
Resumable DDP pre-training script for the selective-SSM (Mamba) legal LM.
Launched by the notebook with:  python -m torch.distributed.run --standalone --nproc_per_node=<GPUs> train_ddp.py config.json
Works with 1 or 2 GPUs (global batch is fixed, grad-accumulation adapts) and on CPU (gloo) for smoke tests.
"""
import os, sys, json, time, math, glob, contextlib
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader

cfg = json.load(open(sys.argv[1]))
sys.path.insert(0, cfg["output_dir"])
from model_ssm import Model3_SSM
from data_ssm import CausalMemmapDataset, rank_micro_batches, val_micro_batches

# ----------------------------------------------------------------------------- distributed setup
rank = int(os.environ.get("RANK", 0))
world = int(os.environ.get("WORLD_SIZE", 1))
local_rank = int(os.environ.get("LOCAL_RANK", 0))
use_cuda = torch.cuda.is_available()
if use_cuda:
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)
else:
    device = torch.device("cpu")
if world > 1:
    dist.init_process_group(backend="nccl" if use_cuda else "gloo")
is_main = rank == 0
use_amp = use_cuda


def log(*a):
    if is_main:
        print(*a, flush=True)


torch.manual_seed(cfg["seed"])
np.random.seed(cfg["seed"])

# ----------------------------------------------------------------------------- hyper-parameters
micro_bs = cfg["micro_batch_size"]
gbs = cfg["global_batch_samples"]
assert gbs % (world * micro_bs) == 0, f"global batch {gbs} not divisible by world({world})*micro({micro_bs})"
accum = gbs // (world * micro_bs)
block = cfg["block_size"]
tokens_per_step = gbs * block

train_ds = CausalMemmapDataset(os.path.join(cfg["tokenized_dir"], "ssm_train_part_*.bin"), block)
val_ds = CausalMemmapDataset(os.path.join(cfg["tokenized_dir"], "ssm_val_part_*.bin"), block)
total_steps = len(train_ds) // gbs
max_steps = total_steps if cfg.get("max_train_steps") is None else min(total_steps, int(cfg["max_train_steps"]))
warmup = int(max(10, min(cfg["max_warmup_steps"], cfg["warmup_fraction"] * max_steps)))
lr_max, lr_min = cfg["learning_rate"], cfg["min_lr"]


def get_lr(step):
    if step < warmup:
        return lr_max * (step + 1) / warmup
    if step >= max_steps:
        return lr_min
    ratio = (step - warmup) / max(1, max_steps - warmup)
    return lr_min + 0.5 * (1.0 + math.cos(math.pi * ratio)) * (lr_max - lr_min)


ckpt_dir = cfg["checkpoint_dir"]
os.makedirs(ckpt_dir, exist_ok=True)
latest_path = os.path.join(ckpt_dir, "latest_checkpoint_ssm.pt")
best_path = os.path.join(ckpt_dir, "best_model_ssm.pt")
status_path = cfg["status_path"]

log(f"[setup] world={world} device={device} micro_bs={micro_bs} accum={accum} global_batch={gbs} samples "
    f"({tokens_per_step:,} tokens/step)")
log(f"[setup] train samples={len(train_ds):,} val samples={len(val_ds):,} -> {max_steps:,} optimizer steps "
    f"(one pass), warmup={warmup}")

# ----------------------------------------------------------------------------- model / optimizer
model = Model3_SSM(**cfg["model"]).to(device)
raw_model = model


def configure_optimizer(m, weight_decay, learning_rate):
    """No weight decay on biases, 1-D tensors (norm gains, D) and A_log."""
    decay, no_decay = [], []
    for name, p in m.named_parameters():
        if not p.requires_grad:
            continue
        if p.dim() < 2 or name.endswith("bias") or "norm" in name or "A_log" in name or name.endswith(".D"):
            no_decay.append(p)
        else:
            decay.append(p)
    groups = [{"params": decay, "weight_decay": weight_decay}, {"params": no_decay, "weight_decay": 0.0}]
    return torch.optim.AdamW(groups, lr=learning_rate, betas=(0.9, 0.95), eps=1e-8)


opt = configure_optimizer(raw_model, cfg["weight_decay"], lr_max)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


def pick_resume():
    """Newest checkpoint (highest step) in the working dir or in any attached input (previous notebook output)."""
    cands = []
    for root in cfg["resume_roots"]:
        cands += glob.glob(os.path.join(root, "**", "latest_checkpoint_ssm.pt"), recursive=True)
    best_p, best_s = None, -1
    for p in sorted(set(cands)):
        try:
            try:
                m = torch.load(p, map_location="cpu", mmap=True, weights_only=False)
            except TypeError:
                m = torch.load(p, map_location="cpu")
            s = int(m.get("step", -1))
            del m
        except Exception as e:  # corrupt / half-written file
            log(f"[resume] skipping unreadable checkpoint {p}: {e}")
            continue
        if s > best_s:
            best_p, best_s = p, s
    return best_p, best_s


global_step = 0
best_val = float("inf")
resume_path, resume_step = pick_resume()
if resume_path is not None:
    ck = torch.load(resume_path, map_location="cpu", weights_only=False)
    raw_model.load_state_dict(ck["model_state"])
    opt.load_state_dict(ck["optimizer_state"])
    if use_amp and ck.get("scaler_state"):
        scaler.load_state_dict(ck["scaler_state"])
    global_step = int(ck["step"])
    best_val = float(ck.get("best_val_loss", float("inf")))
    log(f"[resume] restored step {global_step:,} from {resume_path} (best val so far: {best_val:.4f})")
    del ck
else:
    log("[resume] no checkpoint found -> starting from scratch")

if world > 1:
    model = DDP(model, device_ids=[local_rank] if use_cuda else None)
n_params = sum(p.numel() for p in raw_model.parameters())
log(f"[model] {n_params/1e6:.1f}M parameters")


# ----------------------------------------------------------------------------- helpers
def write_status(finished):
    if not is_main:
        return
    tmp = status_path + ".tmp"
    with open(tmp, "w") as f:
        json.dump({"finished": bool(finished), "step": global_step, "max_steps": max_steps,
                   "best_val_loss": None if best_val == float("inf") else best_val,
                   "updated": time.time()}, f)
    os.replace(tmp, status_path)


def save_checkpoint():
    if not is_main:
        return
    ck = {"model_state": raw_model.state_dict(), "optimizer_state": opt.state_dict(),
          "scaler_state": scaler.state_dict() if use_amp else None,
          "step": global_step, "best_val_loss": best_val, "tokens_seen": global_step * tokens_per_step}
    tmp = latest_path + ".tmp"
    torch.save(ck, tmp)
    os.replace(tmp, latest_path)
    log(f"[ckpt] saved step {global_step:,} -> {latest_path}")


@torch.no_grad()
def evaluate():
    raw_model.eval()
    batches = val_micro_batches(len(val_ds), world, rank, micro_bs, cfg["val_samples"])
    tot = torch.zeros(2, device=device)
    if batches:
        for x, y in DataLoader(val_ds, batch_sampler=batches, num_workers=0):
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                _, l = raw_model(x, y)
            tot[0] += l.float()
            tot[1] += 1
    if world > 1:
        dist.all_reduce(tot)
    raw_model.train()
    return (tot[0] / tot[1]).item() if tot[1] > 0 else float("nan")


# ----------------------------------------------------------------------------- training loop
t_start = time.time()
if global_step >= max_steps:
    log("[train] checkpoint is already at the final step - nothing left to train.")
    write_status(True)
else:
    _, train_batches = rank_micro_batches(len(train_ds), world, rank, micro_bs, accum, cfg["seed"],
                                          start_step=global_step, max_steps=max_steps)
    loader = DataLoader(train_ds, batch_sampler=train_batches, num_workers=cfg["num_workers"], pin_memory=use_cuda)
    it = iter(loader)
    model.train()
    opt.zero_grad(set_to_none=True)
    t_last, steps_since_log = time.time(), 0
    finished = False
    log(f"[train] starting at step {global_step:,}/{max_steps:,}")

    while global_step < max_steps:
        lr = get_lr(global_step)
        for g in opt.param_groups:
            g["lr"] = lr

        loss_sum = torch.zeros((), device=device)
        for micro in range(accum):
            x, y = next(it)
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            sync_ctx = contextlib.nullcontext() if (world == 1 or micro == accum - 1) else model.no_sync()
            with sync_ctx:
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                    _, loss = model(x, y)
                scaler.scale(loss / accum).backward()
            loss_sum += loss.detach().float()
            del loss, _

        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        global_step += 1
        steps_since_log += 1

        if global_step % cfg["log_every"] == 0 or global_step == 1:
            avg = loss_sum / accum
            if world > 1:
                dist.all_reduce(avg)
                avg = avg / world
            v = avg.item()
            if not math.isfinite(v):
                log(f"[train] NON-FINITE loss at step {global_step} - aborting (last good checkpoint is kept).")
                sys.exit(2)
            now = time.time()
            sps = (now - t_last) / max(1, steps_since_log)
            eta_h = sps * (max_steps - global_step) / 3600
            mem = f" | mem {torch.cuda.max_memory_allocated()/2**30:.1f}GB" if use_cuda else ""
            log(f"step {global_step:6d}/{max_steps} | loss {v:.4f} | lr {lr:.2e} | gnorm {gnorm.item():.2f} | "
                f"{sps:.2f}s/step | {tokens_per_step/sps/1e3:.1f}k tok/s | elapsed {(now-t_start)/3600:.2f}h | "
                f"ETA {eta_h:.2f}h{mem}")
            t_last, steps_since_log = now, 0

        stop = False
        if global_step % 10 == 0:
            flag = torch.tensor([1.0 if time.time() > cfg["deadline_epoch"] else 0.0], device=device)
            if world > 1:
                dist.all_reduce(flag, op=dist.ReduceOp.MAX)
            stop = flag.item() > 0

        do_eval = (global_step % cfg["eval_every"] == 0) or global_step == max_steps
        if do_eval:
            val = evaluate()
            ppl = math.exp(val) if val < 20 else float("inf")
            log(f"\n[val] step {global_step} | val loss {val:.4f} | ppl {ppl:.2f}\n")
            if is_main and val == val and val < best_val:  # val == val -> not NaN
                best_val = val
                tmp = best_path + ".tmp"
                torch.save({"model_state": raw_model.state_dict(), "step": global_step, "best_val_loss": best_val}, tmp)
                os.replace(tmp, best_path)
                log(f"[val] new best -> {best_path}")
            if world > 1:  # all ranks must agree on best_val for the checkpoint they might write
                bv = torch.tensor([best_val if best_val != float("inf") else 1e30], device=device)
                dist.broadcast(bv, src=0)
                best_val = float(bv.item()) if bv.item() < 1e29 else float("inf")

        if do_eval or stop or global_step % cfg["ckpt_every"] == 0:
            save_checkpoint()
        if global_step == max_steps:
            finished = True
        if stop and not finished:
            log(f"[train] session time budget reached at step {global_step:,} - checkpoint saved, stopping cleanly.")
            break

    write_status(finished)
    log(f"[train] {'FINISHED' if finished else 'PAUSED'} at step {global_step:,}/{max_steps:,} | best val {best_val:.4f}")

if world > 1:
    dist.barrier()
    dist.destroy_process_group()

Writing /kaggle/working/model3_ssm_output/train_ddp.py


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/model3_ssm_output/train_ddp.py'

In [ ]:
import os
import sys
import json
import time
import subprocess

NUM_GPUS = torch.cuda.device_count()
assert NUM_GPUS >= 1, "No GPU found - enable 'GPU T4 x2' under Session options -> Accelerator."
assert GLOBAL_BATCH_SAMPLES % (NUM_GPUS * MICRO_BATCH_SIZE) == 0, \
    f"GLOBAL_BATCH_SAMPLES ({GLOBAL_BATCH_SAMPLES}) must be divisible by NUM_GPUS*MICRO_BATCH_SIZE ({NUM_GPUS * MICRO_BATCH_SIZE})"

DEADLINE_EPOCH = SESSION_START + SESSION_BUDGET_HOURS * 3600
remaining_h = (DEADLINE_EPOCH - time.time()) / 3600
print(f"{NUM_GPUS} GPU(s) | grad-accum per GPU = {GLOBAL_BATCH_SAMPLES // (NUM_GPUS * MICRO_BATCH_SIZE)} | "
      f"time left before the clean-stop deadline: {remaining_h:.1f} h")
if remaining_h < 0.25:
    print("WARNING: almost no time left in this session's budget - training will pause right away. "
          "Start a new session / raise SESSION_BUDGET_HOURS.")

train_cfg = dict(
    seed=SEED, output_dir=OUTPUT_DIR, tokenized_dir=TOKENIZED_DIR, checkpoint_dir=CHECKPOINT_DIR,
    status_path=STATUS_PATH, resume_roots=[OUTPUT_DIR, INPUT_ROOT],
    micro_batch_size=MICRO_BATCH_SIZE, global_batch_samples=GLOBAL_BATCH_SAMPLES, block_size=TRAIN_BLOCK_SIZE,
    learning_rate=LEARNING_RATE, min_lr=MIN_LR, weight_decay=WEIGHT_DECAY,
    warmup_fraction=WARMUP_FRACTION, max_warmup_steps=MAX_WARMUP_STEPS,
    max_train_steps=None,                      # None -> one full pass over the tokenized corpus
    num_workers=DATALOADER_WORKERS, log_every=LOG_EVERY, eval_every=EVAL_EVERY, ckpt_every=CKPT_EVERY,
    val_samples=VAL_SAMPLES, deadline_epoch=DEADLINE_EPOCH, model=MODEL_KWARGS,
)
with open(CONFIG_PATH, "w") as f:
    json.dump(train_cfg, f, indent=2)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"
# env["NCCL_P2P_DISABLE"] = "1"   # uncomment ONLY if the run hangs before step 1 (NCCL peer-to-peer issue)

cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={NUM_GPUS}",
       os.path.join(OUTPUT_DIR, "train_ddp.py"), CONFIG_PATH]
print("Launching:", " ".join(cmd), "\n")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"Training process exited with code {rc} - see the log above. "
                       f"The last good checkpoint (if any) is in {CHECKPOINT_DIR}.")

status = json.load(open(STATUS_PATH)) if os.path.exists(STATUS_PATH) else {}
TRAINING_DONE = bool(status.get("finished", False))
print("\nTRAINING FINISHED - run the evaluation cells below." if TRAINING_DONE else
      f"\nPAUSED at step {status.get('step')}/{status.get('max_steps')}: checkpoint saved. Evaluation is skipped; "
      "start the next session (attach this notebook's output as input) to continue.")

# 10. Test Evaluation and zip submission file

In [ ]:
import sys
sys.path.insert(0, OUTPUT_DIR)
import os
import json
import time
import torch
import torch.nn.functional as F
from tokenizers import Tokenizer
from model_ssm import Model3_SSM

print("=" * 70)
print("INFERENCE & SUBMISSION GENERATION")
print("=" * 70)

# The tokenizer lives in memory only if the tokenizer cell ran in this kernel; otherwise load the saved one.
if "tokenizer" not in globals():
    tokenizer = Tokenizer.from_file(find_artifact("tokenizer_ssm.json"))
BOS_ID, EOS_ID = tokenizer.token_to_id("<bos>"), tokenizer.token_to_id("<eos>")

# -----------------------------------------------------------------------------
# Only evaluate once pre-training has actually finished (a paused run just skips this cell)
# -----------------------------------------------------------------------------
_status = json.load(open(STATUS_PATH)) if os.path.exists(STATUS_PATH) else {}
TRAINING_DONE = bool(_status.get("finished", False))

if not TRAINING_DONE:
    print(f"Pre-training is not finished yet (step {_status.get('step')}/{_status.get('max_steps')}). "
          "Skipping evaluation - continue training in the next session (attach this notebook's output as input).")
else:
    # -------------------------------------------------------------------------
    # Load best weights (fall back to the latest checkpoint if the best-model file is not in this session's output)
    # -------------------------------------------------------------------------
    _best = BEST_MODEL_PATH if os.path.exists(BEST_MODEL_PATH) else (
        find_artifact("best_model_ssm.pt") or find_artifact("latest_checkpoint_ssm.pt"))
    assert _best is not None, "No trained weights found."
    ckpt = torch.load(_best, map_location="cpu", weights_only=False)
    eval_model = Model3_SSM(**MODEL_KWARGS).to(device)
    eval_model.load_state_dict(ckpt["model_state"])
    eval_model.eval()
    print(f"Loaded weights from {_best} (step {ckpt.get('step', 0)})")
    del ckpt

    # -------------------------------------------------------------------------
    # Test file: the course template stores it as {id: {"text": prompt}}; a plain list of records also works.
    # -------------------------------------------------------------------------
    def load_test_items(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, dict):
            return [(str(k), v.get("text", "") if isinstance(v, dict) else str(v)) for k, v in data.items()]
        return [(str(r.get("id", i)), r.get("text", r.get("prompt", ""))) for i, r in enumerate(data)]

    test_items = load_test_items(test_file_path)
    print(f"Loaded {len(test_items):,} test prompts.")

    # -------------------------------------------------------------------------
    # Test perplexity on the prompt texts (first TRAIN_BLOCK_SIZE tokens of every record).
    #   all tokens : every token is scored given everything before it (standard causal-LM perplexity)
    #   2nd half   : only the positions the encoder-decoder notebook scores (tokens[half+1:]), for a like-for-like comparison
    # -------------------------------------------------------------------------
    @torch.no_grad()
    def evaluate_test_ppl(model, items):
        tot_loss = tot_tok = half_loss = half_tok = 0.0
        for _, text in items:
            if not text:
                continue
            ids = tokenizer.encode(text).ids
            if len(ids) < 4:
                continue
            ids = ids[:TRAIN_BLOCK_SIZE]
            seq = [BOS_ID] + ids
            x = torch.tensor([seq[:-1]], dtype=torch.long, device=device)
            y = torch.tensor([seq[1:]], dtype=torch.long, device=device)     # y[j] == ids[j]
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                logits, _ = model(x)
            tok_loss = F.cross_entropy(logits[0].float(), y[0], reduction="none")
            tot_loss += tok_loss.sum().item(); tot_tok += tok_loss.numel()
            second = tok_loss[len(ids) // 2 + 1:]
            half_loss += second.sum().item(); half_tok += second.numel()

        def to_ppl(l, n):
            if n == 0:
                return float("inf"), float("inf")
            avg = l / n
            return avg, (math.exp(avg) if avg < 20 else float("inf"))
        return to_ppl(tot_loss, tot_tok), to_ppl(half_loss, half_tok)

    (test_loss, test_ppl), (half_loss, half_ppl) = evaluate_test_ppl(eval_model, test_items)
    print("\n" + "=" * 50)
    print(f"  Model 3 Test Loss / PPL (all tokens)        : {test_loss:.4f} / {test_ppl:.2f}")
    print(f"  Model 3 Test Loss / PPL (2nd-half positions): {half_loss:.4f} / {half_ppl:.2f}")
    print("=" * 50 + "\n")
    with open(METRICS_PATH, "w") as f:
        json.dump({"test_loss": test_loss, "test_ppl": test_ppl,
                   "test_loss_second_half": half_loss, "test_ppl_second_half": half_ppl}, f, indent=2)

    # -------------------------------------------------------------------------
    # Generation: temperature + top-p sampling, stops once TARGET_GENERATION_WORDS words exist (<eos> is blocked so the
    # text cannot end early), and the
    # output is cut to exactly that many words (the task asks for 128 WORDS, not tokens).
    # The prompt keeps its END if it is too long (the continuation has to follow the last words of the prompt).
    # The whole context is re-run for every new token, so this is the slow part of the notebook.
    # -------------------------------------------------------------------------
    MAX_PROMPT_TOKENS = TRAIN_BLOCK_SIZE - MAX_GENERATION_TOKENS - 1

    def sample_next(logits, temperature, top_p):
        probs = F.softmax(logits.float() / max(temperature, 1e-5), dim=-1)
        sorted_p, sorted_i = torch.sort(probs, descending=True, dim=-1)
        drop = (sorted_p.cumsum(-1) - sorted_p) > top_p      # smallest set whose mass reaches top_p (top token always kept)
        sorted_p = sorted_p.masked_fill(drop, 0.0)
        sorted_p = sorted_p / sorted_p.sum(-1, keepdim=True)
        return sorted_i.gather(-1, torch.multinomial(sorted_p, 1))

    @torch.no_grad()
    def generate_completion(model, prompt):
        model.eval()
        seq = [BOS_ID] + tokenizer.encode(prompt).ids[-MAX_PROMPT_TOKENS:]
        prompt_len = len(seq)
        for step in range(MAX_GENERATION_TOKENS):
            x = torch.tensor([seq[-TRAIN_BLOCK_SIZE:]], dtype=torch.long, device=device)
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                logits, _ = model(x, last_only=True)
            step_logits = logits[:, -1, :].float()
            step_logits[:, :4] = float("-inf")     # <pad>/<bos>/<eos>/<unk> are never generated: always produce the full 128 words
            nxt = sample_next(step_logits, TEMPERATURE, TOP_P).item()
            seq.append(nxt)
            if (step + 1) % 16 == 0 and len(tokenizer.decode(seq[prompt_len:]).split()) >= TARGET_GENERATION_WORDS:
                break
        return " ".join(tokenizer.decode(seq[prompt_len:]).split()[:TARGET_GENERATION_WORDS])

    # Partial results are saved every 25 prompts; a new session picks them up and only generates what is missing.
    predictions = {}
    _partial = find_artifact(os.path.basename(PARTIAL_PATH))
    if _partial:
        with open(_partial, "r", encoding="utf-8") as f:
            predictions = json.load(f)
        print(f"Resuming generation: {len(predictions):,} predictions already done ({_partial}).")

    todo = [(i, t) for i, t in test_items if i not in predictions]
    torch.manual_seed(SEED)          # sampled completions are reproducible
    print(f"Generating completions for {len(todo):,} test prompts...")
    t0 = time.time()
    for n, (doc_id, prompt) in enumerate(todo, 1):
        predictions[doc_id] = {"text": generate_completion(eval_model, prompt)}
        if n <= 2:
            print(f"\n--- Sample {n} ---\nPrompt   : {prompt[:150]}...\nGenerated: {predictions[doc_id]['text'][:200]}")
        if n % 25 == 0 or n == len(todo):
            el = time.time() - t0
            print(f"  {len(predictions):,}/{len(test_items):,} done | {el / n:.1f} s/prompt | ETA {(len(todo) - n) * el / n / 60:.0f} min", flush=True)
            with open(PARTIAL_PATH, "w", encoding="utf-8") as f:
                json.dump(predictions, f, ensure_ascii=False)

    ordered = {i: predictions[i] for i, _ in test_items}          # same order as the test file
    with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
        json.dump(ordered, f, indent=2, ensure_ascii=False)
    print(f"\nEvaluation complete! Saved {len(ordered):,} predictions -> {PREDICTION_PATH}")

In [ ]:
import zipfile

# -----------------------------------------------------------------------------
# Submission packaging: zip ONLY the prediction file (stored as prediction.json inside the archive)
# -----------------------------------------------------------------------------
if os.path.exists(PREDICTION_PATH):
    print(f"Zipping prediction file from: {PREDICTION_PATH}...")
    with zipfile.ZipFile(SUBMISSION_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(PREDICTION_PATH, arcname="prediction.json")

    zip_size_mb = os.path.getsize(SUBMISSION_ZIP_PATH) / (1024 ** 2)
    print("\n" + "=" * 50)
    print("Model 3 Submission Package Complete!")
    print(f"  - Archive Path : {SUBMISSION_ZIP_PATH}")
    print(f"  - Included File: prediction.json")
    print(f"  - Archive Size : {zip_size_mb:.2f} MB")
    print("=" * 50)
else:
    print(f"No prediction file at {PREDICTION_PATH} yet - it is written by the evaluation cell once pre-training has finished.")

### -----------------------------------COMPLETE-------------------------------------